In [1]:
year = int
month = int

In [2]:
# Parameters
year = 2001
month = 10


In [3]:
from pathlib import Path
import xarray as xr
import numpy as np
import os,subprocess, sys

In [4]:
mesh_path = "/work/bk1450/b383184/Amazon/Mercator/data/Zgr_cmesh2.nc"
W_path = f"/work/bk1450/b383184/Amazon/Mercator/data/variables_c/UVW/W_{year}-{month:02d}c.nc"
SSH_path = f"/work/bk1450/b383184/Amazon/Mercator/data/variables_c/tracers/SSH_{year}-{month:02d}c.nc"

outpath = '/work/bk1450/b383184/Amazon/Mercator/data/variables_c/UVW/'

In [5]:
## mesh load
ds_mesh = xr.open_dataset(mesh_path, chunks={})
ds_mesh = ds_mesh.assign_coords(x=np.arange(ds_mesh.sizes["x"]))
ds_mesh = ds_mesh.assign_coords(y=np.arange(ds_mesh.sizes["y"]))
ds_mesh = ds_mesh.assign_coords(z=np.arange(ds_mesh.sizes["z"]))
ds_mesh = ds_mesh.squeeze()

In [6]:
#Construct W depth for each water filled cell
e3t_full = xr.where(
    (ds_mesh.z + 1) <= (ds_mesh.mbathy - 1),  # wet cells above bottom cells
    ds_mesh.e3t_0,  # fill with basin wide e3t for level,
    ds_mesh.e3t_ps,  # add partial cell height otherwise
).where((ds_mesh.z + 1) <= ds_mesh.mbathy)  # remove all non-wet cells below

In [7]:
#W depths (top of cell) as vertical sum of the e3t (including the partially filled last cell above the bottom). 
#Then rename the dimension z coming from the mesh file to depthw which we'll need to align with the W file later.

depthw_ps = e3t_full.cumsum("z").where((ds_mesh.z + 1) <= ds_mesh.mbathy)
depthw_ps = depthw_ps.shift(z=1).fillna(0.0)
depthw_ps = depthw_ps.where(ds_mesh.z <= ds_mesh.mbathy)
depthw_ps = depthw_ps.rename({"z": "depthw"}).drop("depthw")

/tmp/ipykernel_2800867/2567791241.py:7: DeprecationWarning: dropping variables using `drop` is deprecated; use drop_vars.
  depthw_ps = depthw_ps.rename({"z": "depthw"}).drop("depthw")


In [8]:
## Calc height of water colum (relto $\eta=0$)
H_bottom = e3t_full.sum("z")

In [9]:
## load W file
ds_W = xr.open_dataset(W_path, chunks={"time_counter": 1})
ds_W = ds_W.assign_coords(x=np.arange(ds_W.sizes["x"]))
ds_W = ds_W.assign_coords(y=np.arange(ds_W.sizes["y"]))
ds_W = ds_W.assign_coords(z=-depthw_ps, H=H_bottom)

In [10]:
## Load SSH (we need $\eta$)
ds_SSH = xr.open_dataset(SSH_path, engine="netcdf4",chunks={"time_counter": 1})
ds_SSH = ds_SSH.assign_coords(x=np.arange(ds_SSH.sizes["x"]))
ds_SSH = ds_SSH.assign_coords(y=np.arange(ds_SSH.sizes["y"]))

/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/xarray/conventions.py:204: SerializationWarning: variable 'sossheig' has multiple fill values {np.float64(9.96921e+36), np.float32(9.96921e+36)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)


## Correcting W for fixed sea surface:

$z$ is positive upward, $H$ is positive, and $\eta$ is positive upward

$$W_{merc}(z) = W_\eta(z) + W_{fixed}(z)$$

Normally,
$$W_\eta(z) = \frac{z+H}{\eta+H} \frac{\partial \eta}{\partial t}$$
but we register it to the surface and set.
$$W_\eta(z) \equiv \frac{z+H}{H} \frac{\partial \eta}{\partial t}$$
(Note the change in the denominator.)

Then
$$W_{fixed}(z) = W_{merc}(z) - W_\eta(z)$$
and
$$W_{fixed}(0) = W_{fixed}(-H) = 0$$

In [11]:
deta_dt = ds_W.vovecrtz.isel(depthw=0, drop=True)

In [12]:
eta = ds_SSH.sossheig

In [13]:
W_merc = ds_W.vovecrtz.rename("W_merc").fillna(0.0)

In [14]:
W_eta = (ds_W.z + ds_W.H) / (ds_W.H) * deta_dt
W_eta = W_eta.rename("W_eta")
W_eta = W_eta.assign_coords(z=ds_W.z)

In [15]:
W_fixed = W_merc - W_eta
W_fixed = W_fixed.rename("W_fixed")

In [16]:
## old coords
x = xr.open_dataset('/work/bk1450/b383184/Amazon/Mercator/data/variables_c/UVW/U_1993-01c.nc', chunks={}).x
y = xr.open_dataset('/work/bk1450/b383184/Amazon/Mercator/data/variables_c/UVW/U_1993-01c.nc', chunks={}).y

In [17]:
#re-define the x and y to match with the U and V
W_fixed = W_fixed.assign_coords(x=x)
W_fixed = W_fixed.assign_coords(y=y)
W_fixed.name = "vovecrtz"
W_fixed = W_fixed.to_dataset()
W_fixed

<xarray.Dataset> Size: 8GB
Dimensions:       (time_counter: 31, y: 499, x: 1260, depthw: 50)
Coordinates:
  * time_counter  (time_counter) datetime64[ns] 248B 2001-10-01T12:00:00 ... ...
    nav_lon       (y, x) float32 3MB dask.array<chunksize=(499, 1260), meta=np.ndarray>
    nav_lat       (y, x) float32 3MB dask.array<chunksize=(499, 1260), meta=np.ndarray>
  * depthw        (depthw) float32 200B 0.0 1.011 2.086 ... 5.052e+03 5.5e+03
    z             (depthw, y, x) float64 251MB dask.array<chunksize=(50, 499, 1260), meta=np.ndarray>
    H             (y, x) float64 5MB dask.array<chunksize=(499, 1260), meta=np.ndarray>
  * x             (x) float64 10kB 2.306e+03 2.307e+03 ... 3.564e+03 3.565e+03
  * y             (y) float64 4kB 1.375e+03 1.376e+03 ... 1.872e+03 1.873e+03
Data variables:
    vovecrtz      (time_counter, depthw, y, x) float64 8GB dask.array<chunksize=(1, 1, 208, 1260), meta=np.ndarray>

In [18]:
import calendar
import datetime
from datetime import date
import pandas as pd

last_day = calendar.monthrange(year, month)[1]
start_date = datetime.datetime(year, month, 1)
end_date = datetime.datetime(year, month, last_day)


## days in GLORYS
def glorys_days(start_date, end_date):
    return pd.date_range(start=start_date, end=end_date, freq="D") + pd.Timedelta(hours=12)

days = glorys_days(start_date.strftime("%Y-%m-%d")
                   , end_date.strftime("%Y-%m-%d"))
#starts and ends
starts = days[0::2]
ends = days[1::2].tolist()  
ends[-1] = days[-1]

In [19]:
W_fixed = W_fixed.astype('float32')

W_out = f'W_{start_date.strftime("%Y-%m")}fc.nc'

In [20]:
W_fixed.to_netcdf(outpath+W_out)#outpath